[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mickpletcher/finetuning-ai-models/blob/main/04-your-first-finetune/04-your-first-finetune.ipynb)

# 04 Your First Finetune

## Overview

This is the first end to end adapter fine tune in the repo.
You will prepare the sample dataset, attach LoRA adapters, run a short supervised fine tuning job, save the adapter, and compare base output to adapted output.

Estimated time: 35 to 60 minutes on a free GPU. CPU mode is supported for setup and walkthrough, but it is much slower.

## Prerequisites

Complete notebooks 00 through 03 first.
You should already understand the dataset format and basic LoRA settings.

In [ ]:
%pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 datasets==2.19.0 torch==2.2.2 accelerate==0.29.3

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

## CPU And GPU Notes

The notebook always runs top to bottom.
On CPU it keeps the run very small so the workflow stays understandable.
On a free Colab GPU you can let the short training pass run normally.

In [ ]:
from pathlib import Path
import sys
import json

sys.path.append(str(Path("..").resolve()))

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTTrainer
from utils.helpers import CHECKPOINTS_DIR, DATASETS_DIR, LORA_ADAPTER_DIR, format_alpaca, print_trainable_parameters

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_dir = LORA_ADAPTER_DIR
output_dir.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_rows = [
    json.loads(line)
    for line in (DATASETS_DIR / "sample_dataset.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
train_rows = raw_rows[:8] if device == "cuda" else raw_rows[:2]

formatted_records = [
    {
        "text": format_alpaca(item["instruction"], item["input"], item["output"])
    }
    for item in train_rows
]

train_dataset = Dataset.from_list(formatted_records)
train_dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
print_trainable_parameters(model)

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=256,
    args=training_args,
)

In [ ]:
run_training = device == "cuda"

if run_training:
    trainer.train()
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved adapter to {output_dir}")
else:
    print("CPU mode detected. Training is skipped in this notebook. Use a CUDA GPU or Colab to create a real adapter before running the comparison below.")

In [ ]:
def generate_text(current_model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = current_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

test_prompt = "### Instruction:\nExplain why clean training data matters.\n\n### Input:\n\n### Response:\n"

clean_base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_response = generate_text(clean_base_model, test_prompt)

print("Base model response:\n")
print(base_response)

if output_dir.exists() and any(output_dir.iterdir()):
    adapted_base_model = AutoModelForCausalLM.from_pretrained(model_name)
    adapter_loaded = PeftModel.from_pretrained(adapted_base_model, output_dir)
    adapted_response = generate_text(adapter_loaded, test_prompt)
    print("\nAdapter loaded response:\n")
    print(adapted_response)
else:
    print("\nNo trained adapter found yet. Run this notebook on CUDA or in Colab, then rerun this comparison cell.")

## Key Takeaways

* A small clean dataset is enough to learn the full workflow.
* LoRA adapters let you save a lightweight training result.
* CPU mode is useful for setup and inspection.
* A free GPU makes the short training run much more practical.

## Next Steps

Continue to [05 QLoRA Low Resource](../05-qlora-low-resource/05-qlora-low-resource.ipynb).